# Step 0: moving-object masks with SAM 3

XM starts from the masks of everything that moves in the video. For each moving object you give SAM 3 a few clicked points
or a text prompt, SAM 3 tracks it through the whole video, you check the result and save it. The first pipeline step
(`get_depths_clean.py`) stops with a warning while these masks are missing.

Written into the scene folder:
```
<scene>/masks/<object>/<frame>.png   one folder per object (white = object)
<scene>/masks/<frame>.png            all objects combined: what XM reads (rebuilt by every save)
```
The scene is the one set in `config.json` at the repo root. Run this notebook in the `sam3` environment
(`install/sam3.sh`), with access to the gated `facebook/sam3` weights on Hugging Face (`hf auth login`) and a GPU (about
8 GB for 150 frames at 720 px). The click window needs `ipympl` (installed with the `sam3` environment).

## Load the video and SAM 3

In [ ]:
import glob
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

get_ipython().run_line_magic("matplotlib", "widget")   # interactive figures (ipympl): the click window
sys.path.insert(0, os.path.abspath("."))    # XM's Python/ folder (this notebook's folder)
from utils import definitions as D           # the scene from config.json
from sam3.model_builder import build_sam3_video_predictor


def _numeric_key(p):                        # XM's frame order (utils.track_tapnet._numeric_key)
    s = Path(p).stem
    try:
        return int(s)
    except ValueError:
        return s


FRAMES = sorted(glob.glob(str(D.video_dir / D.image_pattern)), key=_numeric_key)
assert FRAMES, f"no frames matching {D.image_pattern} in {D.video_dir}"
T = len(FRAMES)
W, H = Image.open(FRAMES[0]).size

# SAM 3 reads a folder of <index>.<ext> frames: link XM's frames under that naming (in outputs/, the images stay untouched)
LINKS = D.output_dir / "sam3_frames"
LINKS.mkdir(parents=True, exist_ok=True)
for l in LINKS.iterdir():
    l.unlink()
for i, f in enumerate(FRAMES):
    (LINKS / f"{i:05d}{Path(f).suffix.lower()}").symlink_to(os.path.abspath(f))

torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
predictor = build_sam3_video_predictor()
session = predictor.handle_request(dict(type="start_session", resource_path=str(LINKS), offload_video_to_cpu=True))["session_id"]
print(f"{T} frames of {W} x {H} from {D.video_dir}\nmasks go to {D.mask_dir}")

In [ ]:
OBJECTS = {}        # name -> prompts and the tracked masks (T, H, W)
COLORS = plt.cm.tab10.colors


def _track(points=None, text=None, frame=0):
    """One SAM 3 run on the whole video. points: {frame: (xy list, label list)} for one object;
    text: a concept prompt (every instance becomes its own id). Returns {id: (T, H, W) bool}."""
    predictor.handle_request(dict(type="reset_session", session_id=session))
    if text is not None:
        predictor.handle_request(dict(type="add_prompt", session_id=session, frame_index=int(frame), text=text))
        start = int(frame)
    else:
        for t, (xy, lab) in sorted(points.items()):
            predictor.handle_request(dict(type="add_prompt", session_id=session, frame_index=int(t), obj_id=1,
                                          points=[[x / W, y / H] for x, y in xy], point_labels=list(lab)))
        start = min(points)
    out = {}
    for r in predictor.handle_stream_request(dict(type="propagate_in_video", session_id=session,
                                                  propagation_direction="both", start_frame_index=start)):
        o = r["outputs"]
        for oid, m in zip(np.asarray(o["out_obj_ids"]).ravel(), np.asarray(o["out_binary_masks"])):
            out.setdefault(int(oid), np.zeros((T, H, W), bool))[r["frame_index"]] = m
    return out


def track_points(name, frame, points, labels=None):
    """Track one object from points clicked on `frame`: points [(x, y), ...] in pixels, labels 1 = object, 0 = not the
    object (default: all 1). Calling it again with the same name on another frame adds those points (fix where it fails)."""
    obj = OBJECTS.get(name, {})
    prompts = obj.get("points", {})
    prompts[int(frame)] = (list(points), [1] * len(points) if labels is None else list(labels))
    masks = _track(points=prompts).get(1, np.zeros((T, H, W), bool))
    OBJECTS[name] = dict(points=prompts, masks=masks)
    preview(name)


def track_text(name, text, frame=0, keep=None):
    """Track every instance of a concept ("person", "hand", "dog", ...) found from `frame` on. All instances form one
    object unless `keep` lists the instance ids to keep (the preview labels them)."""
    inst = _track(text=text, frame=frame)
    ids = sorted(inst) if keep is None else list(keep)
    masks = np.any([inst[i] for i in ids], 0) if ids else np.zeros((T, H, W), bool)
    OBJECTS[name] = dict(text=text, instances=inst, masks=masks)
    print(f"'{text}': instances {sorted(inst)}, kept {ids}")
    preview(name)


def show(frame=0, name=None, ax=None):
    """Frame with pixel axes (to read off click coordinates), with the object's mask and points if given."""
    ax = ax or plt.subplots(figsize=(10, 10 * H / W))[1]
    ax.imshow(Image.open(FRAMES[frame]))
    if name is not None:
        obj = OBJECTS[name]
        for k, (i, m) in enumerate(sorted(obj.get("instances", {1: obj["masks"]}).items())):
            if m[frame].any():
                ax.imshow(np.dstack([m[frame]] * 3 + [m[frame] * 0.5]) * [*COLORS[k % 10], 1])
                ax.contour(m[frame], levels=[0.5], colors=[COLORS[k % 10]], linewidths=1.5)
                if "instances" in obj:
                    y, x = np.argwhere(m[frame]).mean(0)
                    ax.text(x, y, str(i), color="w", fontsize=12, ha="center", bbox=dict(fc="k", alpha=0.6))
        for (x, y), l in zip(*obj.get("points", {}).get(frame, ([], []))):
            ax.plot(x, y, "o" if l else "x", ms=9, mec="k", mfc="lime" if l else "red", color="red")
    ax.set_title(f"frame {frame}" + (f": {name}" if name else ""))
    return ax


def preview(name, n=6):
    """n evenly spaced frames with the object's mask, and on how many frames it is present."""
    m = OBJECTS[name]["masks"]
    present = m.reshape(T, -1).any(1)
    print(f"{name}: on {present.sum()} / {T} frames" + (f", missing on {np.flatnonzero(~present)[:20].tolist()}" if (~present).any() else ""))
    ts = np.linspace(0, T - 1, n).round().astype(int)
    fig, axs = plt.subplots(1, n, figsize=(3 * n, 3 * H / W + 0.4))
    for ax, t in zip(np.atleast_1d(axs), ts):
        show(t, name, ax)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def preview_video(name, fps=10):
    """Whole video with the mask, written to outputs/sam3_masks_preview/<name>.mp4 (needs opencv)."""
    import cv2
    out = D.output_dir / "sam3_masks_preview"
    out.mkdir(parents=True, exist_ok=True)
    vw = cv2.VideoWriter(str(out / f"{name}.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    for f, m in zip(FRAMES, OBJECTS[name]["masks"]):
        im = cv2.imread(f)
        im[m] = (0.5 * im[m] + [0, 0, 127]).astype(np.uint8)
        vw.write(im)
    vw.release()
    print(out / f"{name}.mp4")


class Clicker:
    """Interactive window to add one object by clicking: click points on the object, press Track, drag the frame
    slider to check the mask on every frame, add points where it fails and Track again, then Save.
    Left click = on the object, right click (or the "not the object" mode) = not the object."""

    def __init__(self, name="object_1", frame=0):
        import ipywidgets as w
        from IPython.display import display
        self.name = w.Text(value=name, description="object")
        self.mode = w.ToggleButtons(options=[("on the object", 1), ("not the object", 0)], value=1)
        self.frame = w.IntSlider(value=frame, min=0, max=T - 1, description="frame", layout=w.Layout(width="60%"))
        buttons = [w.Button(description=d) for d in ("Track", "Undo point", "Clear frame", "Save")]
        for b, fn in zip(buttons, (self.track, self.undo, self.clear, self.save)):
            b.on_click(lambda _, fn=fn: self._safe(fn))
        self.status = w.HTML()
        with plt.ioff():
            self.fig, self.ax = plt.subplots(figsize=(9, 9 * H / W))
        self.fig.canvas.header_visible = False
        self.ax.set_axis_off()
        self.im = self.ax.imshow(np.zeros((H, W, 3), np.uint8))
        self.over = self.ax.imshow(np.zeros((H, W, 4)))
        self.pos, = self.ax.plot([], [], "o", ms=9, mec="k", mfc="lime", ls="")
        self.neg, = self.ax.plot([], [], "X", ms=10, mec="k", mfc="red", ls="")
        self.fig.tight_layout()
        self.fig.canvas.mpl_connect("button_press_event", lambda e: self._safe(self._click, e))
        self.frame.observe(lambda _: self._safe(self.draw), "value")
        self.name.observe(lambda _: self._safe(self._load), "value")
        self._load()
        display(w.VBox([w.HBox([self.name, self.mode]), self.frame, w.HBox(buttons), self.status, self.fig.canvas]))

    def _safe(self, fn, *args):   # widget callbacks have no visible traceback: show errors in the status line
        try:
            fn(*args)
        except Exception as exc:
            self.status.value = f"<b style='color:red'>{type(exc).__name__}: {exc}</b>"

    def _load(self):              # points of the object with this name (a new name starts empty)
        pts = OBJECTS.get(self.name.value, {}).get("points", {})
        self.points = {t: (list(xy), list(lab)) for t, (xy, lab) in pts.items()}
        self.status.value = "click points on the object, then Track"
        self.draw()

    def _click(self, e):
        if e.inaxes is not self.ax or e.xdata is None or getattr(self.fig.canvas.toolbar, "mode", ""):
            return                # outside the image, or zoom / pan active
        xy, lab = self.points.setdefault(self.frame.value, ([], []))
        xy.append((float(e.xdata), float(e.ydata)))
        lab.append(0 if e.button == 3 else self.mode.value)
        self.draw()

    def draw(self):
        t = self.frame.value
        self.im.set_data(np.asarray(Image.open(FRAMES[t]).convert("RGB")))
        over = np.zeros((H, W, 4))
        m = OBJECTS.get(self.name.value, {}).get("masks")
        if m is not None:
            over[m[t]] = (0.1, 0.9, 0.3, 0.45)
        self.over.set_data(over)
        xy, lab = self.points.get(t, ([], []))
        for line, keep in ((self.pos, 1), (self.neg, 0)):
            p = [q for q, l in zip(xy, lab) if l == keep]
            line.set_data([q[0] for q in p], [q[1] for q in p])
        self.ax.set_title(f"frame {t}: {self.name.value}")
        self.fig.canvas.draw_idle()

    def track(self):
        name, pts = self.name.value, {t: v for t, v in self.points.items() if v[0]}
        if not pts:
            self.status.value = "click at least one point on the object first"
            return
        self.status.value = f"tracking {name} through {T} frames ..."
        masks = _track(points=pts).get(1, np.zeros((T, H, W), bool))
        OBJECTS[name] = dict(points={t: (list(xy), list(lab)) for t, (xy, lab) in pts.items()}, masks=masks)
        n = int(masks.reshape(T, -1).any(1).sum())
        self.status.value = (f"<b>{name}</b>: on {n} / {T} frames. Drag the slider to check; where it fails, click more "
                             "points on that frame and Track again. Save when it is right.")
        self.draw()

    def undo(self):
        xy, lab = self.points.get(self.frame.value, ([], []))
        if xy:
            xy.pop(), lab.pop()
        self.draw()

    def clear(self):
        self.points.pop(self.frame.value, None)
        self.draw()

    def save(self):
        name = self.name.value
        if name not in OBJECTS:
            self.status.value = "press Track before Save"
            return
        save(name)
        self.status.value = (f"saved <b>{name}</b> to {D.mask_dir / name}. Next object: type a new name above, "
                             "click on it and Track.")


def save(name):
    """Write masks/<name>/ and rebuild the combined masks/<frame>.png that XM reads."""
    d = D.mask_dir / name
    d.mkdir(parents=True, exist_ok=True)
    for f, m in zip(FRAMES, OBJECTS[name]["masks"]):
        Image.fromarray(m.astype(np.uint8) * 255).save(d / f"{Path(f).stem}.png")
    combine()


def remove(name):
    """Delete a saved object folder and rebuild the combined masks."""
    import shutil
    shutil.rmtree(D.mask_dir / name)
    combine()


def combine():
    """masks/<frame>.png = union of every object folder in masks/ (also ones saved in earlier sessions)."""
    marker = D.mask_dir / ".combined_from_object_folders"
    if glob.glob(str(D.mask_dir / "*.png")) and not marker.exists():
        raise RuntimeError(f"{D.mask_dir} already holds masks that this notebook did not make. Move them into a sub-folder "
                           "(e.g. masks/old/, then they count as one object) or delete them, and save again.")
    objs = sorted(p for p in D.mask_dir.iterdir() if p.is_dir())
    for f in FRAMES:
        u = np.zeros((H, W), bool)
        for o in objs:
            p = o / f"{Path(f).stem}.png"
            if p.exists():
                u |= np.asarray(Image.open(p).convert("L")) > 0
        Image.fromarray(u.astype(np.uint8) * 255).save(D.mask_dir / f"{Path(f).stem}.png")
    marker.write_text("masks/*.png are the union of the object folders, written by make_masks_sam3.ipynb\n")
    print(f"saved; {D.mask_dir} now combines {[o.name for o in objs]}")

## Add objects by clicking: the click window
Running the cell opens the window. For each moving object:
1. type its name in **object**, drag **frame** to a frame where it is clearly visible;
2. click a few points on it (left click = on the object; right click, or the **not the object** mode = background);
3. press **Track**: SAM 3 tracks it through the whole video, the mask shows in green;
4. drag **frame** to check every frame. Where the mask is lost or leaks, click more points on that frame and press
   **Track** again (the points of all frames are used together);
5. press **Save**, then type the next object's name.

Zoom / pan with the toolbar on the left of the figure; switch it off again to click. Typing the points works too:
`track_points("dog", frame=0, points=[(420, 310)], labels=[1])`.

In [ ]:
clicker = Clicker("object_1")

## Add an object by a text prompt
SAM 3 finds and tracks every instance of the concept. All instances form one object; to keep only some, pass
`keep=[ids]` with the ids shown in the preview (e.g. `track_text("left_hand", "hand", keep=[0])`).

In [ ]:
# edit: name and prompt
track_text("people", "person")

## Check and save
`preview_video(name)` writes the whole tracked video for checking. `save(name)` writes `masks/<name>/` and the combined
masks; `remove(name)` deletes a saved object again.

In [ ]:
for name in OBJECTS:
    save(name)

## Done
Close the session to free the GPU. Then run XM as usual; `python check_masks.py` confirms that the masks are found.

In [ ]:
predictor.handle_request(dict(type="close_session", session_id=session))